# Notebook 4 — LLM-as-a-Judge con mitigación de sesgos

**Autor:** Agustín  
**Módulo:** M2  
**Modelo evaluado:** `PlanTL-GOB-ES/roberta-base-biomedical-clinical-es` fine-tuneado con LoRA sobre DisTEMIST  
**Tarea del modelo:** NER de enfermedades en texto clínico en español (token classification, salida BIO)

## Objetivo

Implementar un juez LLM (Llama-3.3-70B vía Groq) sobre un subconjunto de ejemplos «ricos» del test set,
e identificar y mitigar **al menos 2 sesgos conocidos** del juez:

1. **Sesgo de posición**: el juez favorece la respuesta que aparece primero en el prompt.
2. **Sesgo de longitud**: el juez premia respuestas más largas independientemente de su corrección.
3. **Sesgo de auto-preferencia** (documentado): el juez podría favorecer el estilo de output de su propia familia.

El entregable es la **comparación antes/después de la mitigación** con números concretos.

## Contrato de datos (acordado con el equipo)

- `format_gold_example(row)` → devuelve `esperado` como **lista** (`list[str]`), no string.
- `select_rich_examples()` filtra sobre el mismo split `test` que usa Luis para `micro_prf1_by_doc`.
  Esta asimetría se documenta explícitamente: el subset no es comparable directamente al F1 global.
- La conversión lista → string ocurre en el prompt-building con `', '.join(lista)`,
  **no** en `format_gold_example()` — para respetar el contrato de datos del equipo.


## 0. Setup e imports

In [6]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("[INFO] No estamos en Colab — ajusta BASE_DIR manualmente.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import subprocess, sys

pkgs = ["groq", "peft", "datasets", "transformers", "accelerate", "torchao"]
for pkg in pkgs:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", pkg])


In [8]:
import re, json, random, time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from collections import defaultdict

from groq import Groq
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForTokenClassification
from peft import PeftModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Versiones de librerías:")
import transformers as _tr, datasets as _ds, peft as _peft
print(f"  transformers: {_tr.__version__}")
print(f"  datasets:     {_ds.__version__}")
print(f"  peft:         {_peft.__version__}")
print(f"  torch:        {torch.__version__}")
print(f"  numpy:        {np.__version__}")


Versiones de librerías:
  transformers: 5.16.1
  datasets:     4.0.0
  peft:         0.20.0
  torch:        2.11.0+cu128
  numpy:        2.1.3


In [9]:
# ============================================================
#  CONFIGURACIÓN — ajustar solo estas variables
# ============================================================

# Resolución de rutas compatible con Colab y entornos locales
if IN_COLAB:
    _drive_root = Path("/content/drive/MyDrive/TopicosIA")
    if (_drive_root / "Proyecto-Salud" / "M1" / "distemist_final").exists():
        BASE_DIR = _drive_root / "Proyecto-Salud" / "M1"
    else:
        BASE_DIR = _drive_root
else:
    BASE_DIR = Path(".")

SPLITS_DIR = BASE_DIR / "distemist_final"
MODEL_DIR  = BASE_DIR / "saved_models" / "clinical_bert-distemist-lora"

BASE_CHECKPOINT = "PlanTL-GOB-ES/roberta-base-biomedical-clinical-es"

# Juez LLM (Groq — https://console.groq.com/keys)
GROQ_API_KEY = ""   # <-- pegar la API key aquí (gratis en https://console.groq.com/keys) o usar Colab Secrets (GROQ_API_KEY)

# Modelo preferido para el juez.
# Si este modelo no existe o no está habilitado en tu cuenta de Groq (ej. por cambios
# en el catálogo o nivel de cuenta), el notebook seleccionará automáticamente el mejor
# modelo disponible de la lista de respaldo (CANDIDATE_JUDGE_MODELS).
JUDGE_MODEL  = "llama-3.3-70b-versatile"

# Modelos candidatos por orden de preferencia para evaluación LLM
CANDIDATE_JUDGE_MODELS = [
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
    "openai/gpt-oss-120b",
    "llama3-70b-8192",
    "llama-3.2-11b-vision-preview",
    "qwen/qwen3.6-27b",
    "deepseek-r1-distill-llama-70b",
    "mixtral-8x7b-32768",
    "gemma2-9b-it",
    "llama3-8b-8192",
]

MIN_UNIQUE_ENTITIES = 3
N_RICH_EXAMPLES     = 30
JUDGE_TEMPERATURE   = 0.0   # determinístico
JUDGE_MAX_TOKENS    = 1500  # suficiente para modelos con CoT/razonamiento en ejemplos largos

print(f"BASE_DIR:       {BASE_DIR}")
print(f"SPLITS_DIR:     {SPLITS_DIR}")
print(f"MODEL_DIR:      {MODEL_DIR}")
print(f"Juez propuesto: {JUDGE_MODEL}")

# Asegurar que los pesos LoRA estén en Drive (si no existen, se descargan del repo)
if not (MODEL_DIR / "adapter_model.safetensors").exists():
    print(f"[INFO] Modelo LoRA no encontrado en {MODEL_DIR}. Descargando desde rama clinical-bert del repo...")
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    import subprocess, shutil
    subprocess.run([
        "git", "clone", "--depth", "1", "-b", "clinical-bert",
        "https://github.com/luisNP21/Topicos-IA.git", "/tmp/repo_clinical"
    ], check=True)
    for item in Path("/tmp/repo_clinical/M1/clinical_BERT/clinical_bert-distemist-lora").glob("*"):
        shutil.copy2(item, MODEL_DIR)
    print(f"[OK] Modelo LoRA listo en {MODEL_DIR}")
else:
    print(f"[OK] Modelo LoRA verificado en {MODEL_DIR}")


BASE_DIR:       /content/drive/MyDrive/TopicosIA
SPLITS_DIR:     /content/drive/MyDrive/TopicosIA/distemist_final
MODEL_DIR:      /content/drive/MyDrive/TopicosIA/saved_models/clinical_bert-distemist-lora
Juez propuesto: llama-3.3-70b-versatile
[OK] Modelo LoRA verificado en /content/drive/MyDrive/TopicosIA/saved_models/clinical_bert-distemist-lora


In [10]:
import os
from groq import Groq

api_key = GROQ_API_KEY or os.environ.get("GROQ_API_KEY", "")
if not api_key:
    raise ValueError(
        "No se encontró la API key de Groq. "
        "Obtén una gratis en https://console.groq.com/keys y pégala en GROQ_API_KEY."
    )

groq_client = Groq(api_key=api_key)

# 1. Consultar modelos disponibles en la cuenta de Groq
available_model_ids = []
try:
    models_response = groq_client.models.list()
    available_model_ids = [m.id for m in models_response.data]
    print(f"Modelos reportados en tu cuenta de Groq ({len(available_model_ids)}):")
    for mid in available_model_ids[:8]:
        print(f"  - {mid}")
    if len(available_model_ids) > 8:
        print(f"  ... y {len(available_model_ids) - 8} más.")
except Exception as e:
    print(f"[AVISO] No se pudo obtener la lista con models.list(): {e}")

# 2. Función auxiliar para verificar si un modelo puede generar respuestas
def test_model_ping(model_name: str) -> bool:
    try:
        groq_client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": "hola"}],
            max_tokens=10,
        )
        return True
    except Exception:
        return False

# 3. Selección y verificación robusta con fallback automático
chosen_model = None
pool = [JUDGE_MODEL] + [m for m in CANDIDATE_JUDGE_MODELS if m != JUDGE_MODEL]

# Si conocemos los modelos disponibles, priorizamos los que existen en la cuenta
if available_model_ids:
    candidate_list = [m for m in pool if m in available_model_ids]
    # Añadir otros modelos de chat/completions si ninguno de la lista preferida está
    for mid in available_model_ids:
        if mid not in candidate_list and any(k in mid for k in ["llama", "gpt", "qwen", "mixtral", "gemma"]):
            candidate_list.append(mid)
else:
    candidate_list = pool

print(f"\nVerificando conectividad y disponibilidad del modelo juez...")
for cand in candidate_list:
    print(f"  Probando '{cand}'...", end=" ")
    if test_model_ping(cand):
        chosen_model = cand
        print("OK")
        break
    else:
        print("No disponible / error 404")

if not chosen_model:
    raise RuntimeError(
        f"No se pudo inicializar ningún modelo juez en Groq.\n"
        f"Modelos detectados en tu cuenta: {available_model_ids}\n"
        f"Por favor verifica tu API key y los modelos activos en https://console.groq.com/docs/models"
    )

if chosen_model != JUDGE_MODEL:
    print(f"\n[AVISO] El modelo configurado '{JUDGE_MODEL}' no está disponible en tu cuenta/tier.")
    print(f"[OK] Fallback automático aplicado con éxito: usando '{chosen_model}'.")
    JUDGE_MODEL = chosen_model
else:
    print(f"\n[OK] Modelo configurado '{JUDGE_MODEL}' verificado exitosamente.")

# 4. Comprobar si el modelo soporta response_format={'type': 'json_object'}
SUPPORTS_JSON_MODE = True
try:
    groq_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": "Responde solo JSON."},
            {"role": "user", "content": "Genera {\"ping\": \"pong\"}"},
        ],
        max_tokens=25,
        response_format={"type": "json_object"},
    )
    print("  Soporte de JSON mode nativo (response_format): SI")
except Exception:
    SUPPORTS_JSON_MODE = False
    print("  Soporte de JSON mode nativo (response_format): NO (se usará parseo robusto por regex)")

print(f"\nJuez LLM listo para evaluación: {JUDGE_MODEL} (Groq API)")


Modelos reportados en tu cuenta de Groq (14):
  - meta-llama/llama-prompt-guard-2-86m
  - groq/compound
  - qwen/qwen3.8-27b
  - allam-2-7b
  - canopylabs/orpheus-arabic-saudi
  - meta-llama/llama-prompt-guard-2-22m
  - qwen/qwen3.6-27b
  - canopylabs/orpheus-v1-english
  ... y 6 más.

Verificando conectividad y disponibilidad del modelo juez...
  Probando 'openai/gpt-oss-120b'... OK

[AVISO] El modelo configurado 'llama-3.3-70b-versatile' no está disponible en tu cuenta/tier.
[OK] Fallback automático aplicado con éxito: usando 'openai/gpt-oss-120b'.
  Soporte de JSON mode nativo (response_format): NO (se usará parseo robusto por regex)

Juez LLM listo para evaluación: openai/gpt-oss-120b (Groq API)


## 1. Cargar datos y modelo

### 1.1 Helpers reutilizados de M1 (mismas funciones que `clinical_BERT/03_finetuning.ipynb`)


In [11]:
def strip_chunk_suffix(doc_id: str) -> str:
    """Elimina el sufijo _chunkN para obtener el ID del documento original."""
    return re.sub(r"_chunk\d+$", "", doc_id)


def bio_to_entity_set(tokens: list, tags: list) -> set:
    """Convierte secuencia BIO (tokens, tags) a set de strings de entidades."""
    entities, current = [], []
    for tok, tag in zip(tokens, tags):
        if tag == "B-ENFERMEDAD":
            if current:
                entities.append(" ".join(current))
            current = [tok]
        elif tag == "I-ENFERMEDAD" and current:
            current.append(tok)
        else:
            if current:
                entities.append(" ".join(current))
            current = []
    if current:
        entities.append(" ".join(current))
    return set(e.lower().strip() for e in entities)


def aggregate_entities_by_original_doc(doc_ids: list, entity_sets: list) -> dict:
    """Junta los sets de entidades de todos los chunks del mismo documento."""
    grouped = {}
    for doc_id, ents in zip(doc_ids, entity_sets):
        orig_id = strip_chunk_suffix(doc_id)
        grouped.setdefault(orig_id, set()).update(ents)
    return grouped


print("Helpers de M1 cargados.")


Helpers de M1 cargados.


### 1.2 Cargar el eval set (split test)


In [12]:
def load_eval_set(splits_dir: Path):
    """
    Carga el split 'test' del dataset en formato BERT (distemist_bert_format).
    Formato de cada fila: doc_id (str), tokens (list[str]), ner_tags (list[int]).
    """
    dataset_path = splits_dir / "distemist_bert_format"
    if not dataset_path.exists() and (splits_dir / "distemist_final" / "distemist_bert_format").exists():
        dataset_path = splits_dir / "distemist_final" / "distemist_bert_format"
    ds = load_from_disk(str(dataset_path))
    print(f"Splits disponibles: {list(ds.keys())}")
    print(f"Tamaño split 'test': {len(ds['test'])} chunks")
    print(f"Columnas: {ds['test'].column_names}")
    return ds


ds      = load_eval_set(SPLITS_DIR)
test_ds = ds["test"]

try:
    label_names = test_ds.features["ner_tags"].feature.names
except (AttributeError, KeyError):
    label_names = ["O", "B-ENFERMEDAD", "I-ENFERMEDAD"]

id2label    = {i: l for i, l in enumerate(label_names)}
label2id    = {l: i for i, l in id2label.items()}
print("Etiquetas:", id2label)


Splits disponibles: ['train', 'dev', 'test']
Tamaño split 'test': 233 chunks
Columnas: ['doc_id', 'tokens', 'ner_tags']
Etiquetas: {0: 'O', 1: 'B-ENFERMEDAD', 2: 'I-ENFERMEDAD'}


### 1.3 `format_gold_example` — contrato de datos del equipo

> **Contrato:** `format_gold_example()` devuelve SIEMPRE una **lista** (`list[str]`), no un string.
> Si el juez necesita un string legible para el prompt, usar `', '.join(lista)` en el prompt-building.


In [13]:
def format_gold_example(row: dict, id2label: dict) -> list:
    """
    Devuelve la lista de entidades gold de una fila del test set.

    Contrato de datos (acordado con el equipo):
        → Devuelve SIEMPRE list[str], no string.
        → Si el juez necesita un string: usar ', '.join(...) en el prompt-building,
          NO modificar esta función.
    """
    tokens   = row["tokens"]
    ner_tags = row["ner_tags"]
    if ner_tags and isinstance(ner_tags[0], int):
        tag_strings = [id2label[t] for t in ner_tags]
    else:
        tag_strings = ner_tags
    entity_set = bio_to_entity_set(tokens, tag_strings)
    return sorted(entity_set)  # orden determinístico


ejemplo      = test_ds[0]
gold_ejemplo = format_gold_example(ejemplo, id2label)
print(f"doc_id: {ejemplo['doc_id']}")
print(f"Gold (lista): {gold_ejemplo}")
assert isinstance(gold_ejemplo, list), "format_gold_example debe devolver una lista"
print("Contrato verificado: devuelve lista.")


doc_id: es-S0210-56912008000200007-2_chunk0
Gold (lista): ['cid);', 'shock séptico']
Contrato verificado: devuelve lista.


### 1.4 Cargar modelo fine-tuneado y definir inferencia


In [14]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")

# Evitar incompatibilidad de version de torchao en Colab
import sys
import peft.import_utils
from packaging import version
peft.import_utils.TORCHAO_MINIMUM_VERSION = version.parse("0.0.1")
peft.import_utils.is_torchao_available = lambda: False
for mod in list(sys.modules.values()):
    if hasattr(mod, "is_torchao_available"):
        try:
            mod.is_torchao_available = lambda: False
        except Exception:
            pass

tokenizer  = AutoTokenizer.from_pretrained(BASE_CHECKPOINT)
base_model = AutoModelForTokenClassification.from_pretrained(
    BASE_CHECKPOINT,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)
model = PeftModel.from_pretrained(base_model, str(MODEL_DIR))
model = model.to(device)
model.eval()
print(f"Modelo cargado desde: {MODEL_DIR}")


Dispositivo: cuda


config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/540k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  504MB            

model.safetensors: downloading bytes:           |  0.00B            

Modelo cargado desde: /content/drive/MyDrive/TopicosIA/saved_models/clinical_bert-distemist-lora


In [15]:
def run_roberta_inference(row: dict) -> list:
    """
    Pasa una fila por el modelo RoBERTa fine-tuneado.
    Devuelve lista de entidades predichas (misma forma que format_gold_example).
    La conversión BIO -> lista permite presentar el output al juez de forma
    arquitectura-agnóstica (igual que mT5 después de mt5_output_to_entity_set).
    """
    tokens   = row["tokens"]
    encoding = tokenizer(
        tokens, is_split_into_words=True,
        return_tensors="pt", truncation=True, max_length=512,
    )
    word_ids_list = encoding.word_ids(batch_index=0)
    inputs = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
    pred_ids = logits.argmax(dim=-1)[0].tolist()

    word_preds, seen = [], set()
    for idx, wid in zip(pred_ids, word_ids_list):
        if wid is None or wid in seen:
            continue
        word_preds.append(id2label[idx])
        seen.add(wid)

    entity_set = bio_to_entity_set(tokens[: len(word_preds)], word_preds)
    return sorted(entity_set)


pred_ejemplo = run_roberta_inference(test_ds[0])
print(f"Pred (lista): {pred_ejemplo}")
assert isinstance(pred_ejemplo, list)
print("Inferencia OK")


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Pred (lista): ['cid);', 'shock séptico']
Inferencia OK


## 2. Seleccionar ejemplos ricos (rich examples)

Los ejemplos ricos son un **subconjunto** del mismo split `test` que usa Luis para `micro_prf1_by_doc`.

> **Nota explícita (para el reporte / harness de Isa):** el scorecard del juez sobre los rich examples
> NO es directamente comparable al F1 global sobre todo el test set.
> Los rich examples son un subconjunto curado — esta asimetría debe quedar explícita en el reporte.


In [16]:
def select_rich_examples(
    test_dataset, id2label: dict, min_unique_entities: int = 3, n: int = 30,
) -> list:
    """
    Selecciona documentos con >= min_unique_entities entidades gold únicas (por doc original).
    Corre la inferencia del modelo sobre cada documento rico.

    Devuelve lista de dicts:
        doc_id : str
        n_gold : int
        gold   : list[str]  -- contrato de equipo: lista, no string
        pred   : list[str]
        chunks : list[dict] -- filas originales del dataset
    """
    doc_chunks = defaultdict(list)
    for row in test_dataset:
        doc_chunks[strip_chunk_suffix(row["doc_id"])].append(row)

    rich_docs = []
    for orig_id, chunks in doc_chunks.items():
        gold_set = set()
        for chunk in chunks:
            gold_set.update(format_gold_example(chunk, id2label))
        if len(gold_set) >= min_unique_entities:
            rich_docs.append({
                "doc_id": orig_id, "n_gold": len(gold_set),
                "gold": sorted(gold_set), "chunks": chunks,
            })

    rich_docs.sort(key=lambda d: d["n_gold"], reverse=True)
    rich_docs = rich_docs[:n]

    print(f"Corriendo inferencia sobre {len(rich_docs)} documentos ricos...")
    for doc in rich_docs:
        pred_set = set()
        for chunk in doc["chunks"]:
            pred_set.update(run_roberta_inference(chunk))
        doc["pred"] = sorted(pred_set)

    print(f"Documentos ricos seleccionados: {len(rich_docs)}")
    counts = [d["n_gold"] for d in rich_docs]
    print(f"Entidades gold — media: {np.mean(counts):.1f}, máx: {max(counts)}")
    return rich_docs


rich_examples = select_rich_examples(
    test_ds, id2label=id2label,
    min_unique_entities=MIN_UNIQUE_ENTITIES,
    n=N_RICH_EXAMPLES,
)

for doc in rich_examples[:3]:
    print(f"\ndoc_id: {doc['doc_id']} | n_gold: {doc['n_gold']}")
    print(f"  Gold: {doc['gold']}")
    print(f"  Pred: {doc['pred']}")


Corriendo inferencia sobre 30 documentos ricos...
Documentos ricos seleccionados: 30
Entidades gold — media: 18.0, máx: 28

doc_id: es-S0376-78922009000100011-1 | n_gold: 28
  Gold: ['adenopatías', 'adenopatías bilaterales en las cadenas iliacas interna y externa y en las cadenas inguinales;', 'adenopatías bilaterales ilíacas e inguinales', 'afectación de las cadenas ganglionares', 'brucelosis,', 'carcinoma verrugoso de buscke-lowenstein.', 'condiloma acuminado gigante de buschke-lowenstein.', 'herida de la incisión de linfadenectomía', 'hipogonadismo hipergonadotrópico,', 'infiltraba igualmente los tejidos pubianos, el escroto', 'infiltración de la grasa del tejido celular subcutáneo de la pared interna de los muslos ni de la grasa del periné;', 'infiltración metastásica.', 'infiltrar también la fascia del músculo aductor', 'lesiones óseas en las ramas isquio-ileo-pubianas', 'lesión', 'lesión penoescrotal', 'lesión tumoral;', 'linfogranuloma venéreo.', 'lúes', 'masa tumoral,', 'metást

## 3. RUBRICA del juez

Evalúa en 4 dimensiones (1–5 cada una):

| Dimensión | Descripción |
|---|---|
| **Completitud** | ¿Capturó la mayoría de las enfermedades mencionadas? |
| **Exactitud de boundary** | ¿Los nombres coinciden con el gold (o casi)? |
| **Relevancia clínica** | ¿Las predicciones son términos clínicamente válidos? |
| **Ausencia de ruido** | ¿Evitó marcar términos que NO son enfermedades? |

Score final = promedio de las 4 dimensiones.


In [17]:
RUBRICA_SYSTEM = (
    "Eres un médico especialista en terminología médica y anotación de entidades clínicas. "
    "Tu tarea es evaluar la calidad de las entidades clínicas predichas por un sistema de NER, "
    "comparando sus predicciones con una lista gold. "
    "Evalúa SOLO el contenido. No sabes qué modelo generó las predicciones."
)

RUBRICA_TEMPLATE = """
# Evaluación de NER clínico — RUBRICA

## Referencia (gold standard)
Enfermedades correctas: {gold_str}

## Predicción del sistema
Enfermedades detectadas: {pred_str}

## Instrucciones
Evalúa en cada dimensión del 1 al 5:

1. **Completitud** (¿capturó la mayoría de las enfermedades gold?):
   5=todas/casi todas | 3=la mitad | 1=prácticamente nada

2. **Exactitud de boundary** (¿los nombres coinciden con el gold?):
   5=coincidencia exacta o diferencia mínima | 3=algunos coinciden, otros truncados | 1=no se parecen

3. **Relevancia clínica** (¿las predicciones son términos de enfermedades válidos?):
   5=todas válidas | 3=mezcla | 1=mayoría inválidas

4. **Ausencia de ruido** (¿evitó marcar términos que NO son enfermedades?):
   5=sin FP notables | 3=algunos FP | 1=demasiados FP

## Respuesta
Responde ÚNICAMENTE en JSON válido con el siguiente formato:
```json
{{
  "completitud": <1-5>,
  "exactitud_boundary": <1-5>,
  "relevancia_clinica": <1-5>,
  "ausencia_ruido": <1-5>,
  "justificacion": "<una oración breve>"
}}
```
"""


def build_judge_prompt(gold: list, pred: list, order: str = "normal") -> str:
    gold_str = ", ".join(gold) if gold else "(ninguna)"
    pred_str = ", ".join(pred) if pred else "(ninguna)"
    if order == "inverted":
        return RUBRICA_TEMPLATE.format(gold_str=pred_str, pred_str=gold_str)
    return RUBRICA_TEMPLATE.format(gold_str=gold_str, pred_str=pred_str)


def parse_judge_response(text: str) -> dict:
    # 1. Intentar parsear bloque JSON completo
    for pattern in [r"```(?:json)?\s*({[\s\S]*?})\s*```", r"({[\s\S]*?})"]:
        m = re.search(pattern, text)
        if m:
            try:
                return json.loads(m.group(1))
            except Exception:
                pass
    try:
        return json.loads(text.strip())
    except Exception:
        pass

    # 2. Fallback resiliente: extracción directa por regex de cada dimensión (evita fallos por CoT o truncamiento)
    result = {}
    for key in ["completitud", "exactitud_boundary", "relevancia_clinica", "ausencia_ruido"]:
        val_m = re.search(rf'"{key}"\s*:\s*([1-5])', text, re.IGNORECASE)
        result[key] = int(val_m.group(1)) if val_m else None

    just_m = re.search(r'"justificacion"\s*:\s*"([^"\n]+)', text, re.IGNORECASE)
    result["justificacion"] = just_m.group(1) if just_m else (text[-200:].strip() if text else "Sin texto")
    return result


def compute_score(d: dict) -> float:
    vals = [d.get(k) for k in ["completitud","exactitud_boundary",
                                "relevancia_clinica","ausencia_ruido"]
            if d.get(k) is not None]
    return float(np.mean(vals)) if vals else None


def call_judge(gold: list, pred: list, order: str = "normal",
               retry: int = 5, sleep_s: float = 1.0) -> dict:
    """
    Llama al juez LLM vía Groq con reintentos y tolerancia a fallos.
    Si el modelo no soporta response_format={'type': 'json_object'}, hace fallback
    automático a modo texto estándar y parsea la respuesta con regex.
    Maneja también esperas exponenciales ante rate limits (429).
    """
    prompt = build_judge_prompt(gold, pred, order=order)
    use_json_mode = globals().get("SUPPORTS_JSON_MODE", True)

    for attempt in range(retry):
        try:
            kwargs = {
                "model": JUDGE_MODEL,
                "messages": [
                    {"role": "system", "content": RUBRICA_SYSTEM},
                    {"role": "user", "content": prompt},
                ],
                "temperature": JUDGE_TEMPERATURE,
                "max_tokens": JUDGE_MAX_TOKENS,
            }
            if use_json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            response = groq_client.chat.completions.create(**kwargs)
            time.sleep(sleep_s)
            content = response.choices[0].message.content
            parsed = parse_judge_response(content)

            # Si se obtuvo al menos una dimensión numérica válida, el llamado fue exitoso
            if any(parsed.get(k) is not None for k in ["completitud", "exactitud_boundary", "relevancia_clinica", "ausencia_ruido"]):
                return parsed

            # Si no parseó pero usábamos json_mode, reintentar sin json_mode
            if attempt == 0 and use_json_mode:
                use_json_mode = False
                continue

            return parsed
        except Exception as e:
            err_str = str(e)
            if "response_format" in err_str or "json_object" in err_str:
                use_json_mode = False

            is_rate_limit = "429" in err_str or "rate_limit" in err_str.lower()
            base_wait = 5 if is_rate_limit else 2
            wait_time = base_wait * (attempt + 1)
            print(f"    [Reintento {attempt+1}/{retry}] {e} -> esperando {wait_time}s...")
            time.sleep(wait_time)

    return {"completitud": None, "exactitud_boundary": None,
            "relevancia_clinica": None, "ausencia_ruido": None,
            "justificacion": "JUDGE_CALL_FAILED"}


test_res = call_judge(["asma crónica"], ["asma"])
print("Test call_judge verificado:", test_res)


Test call_judge verificado: {'completitud': 3, 'exactitud_boundary': 3, 'relevancia_clinica': 5, 'ausencia_ruido': 5, 'justificacion': "Capturó 'asma' correctamente pero omitió el modificador 'crónica', sin errores adicionales."}


## 4. Mitigación del sesgo de posición

### Qué es el sesgo de posición
El juez puede favorecer la información que aparece **primero** en el prompt independientemente de su calidad.

### Protocolo de detección
Cada ejemplo se evalúa **dos veces**:
- `order="normal"` — gold como referencia, pred como predicción (rol estándar)
- `order="inverted"` — invertimos roles: si el juez tiene sesgo, el score cambiará al poner pred en el rol de referencia

**Delta de posición** = `|score_normal − score_inverted|`

### Estrategia de mitigación
Reportar el **promedio** de `score_normal` y `score_inverted` como score final.


In [18]:
print(f"Corriendo juez (normal + invertido) sobre {len(rich_examples)} rich examples...")
print(f"Total llamadas al juez: {len(rich_examples) * 2}\n")

position_results = []

for i, doc in enumerate(rich_examples):
    print(f"  [{i+1}/{len(rich_examples)}] {doc['doc_id']} | n_gold={doc['n_gold']} | n_pred={len(doc['pred'])}")

    result_normal    = call_judge(doc["gold"], doc["pred"], order="normal")
    score_normal     = compute_score(result_normal)

    result_inverted  = call_judge(doc["gold"], doc["pred"], order="inverted")
    score_inverted   = compute_score(result_inverted)

    delta = abs(score_normal - score_inverted) if (score_normal is not None and score_inverted is not None) else None
    valid_scores = [s for s in [score_normal, score_inverted] if s is not None]
    score_mitigado = float(np.mean(valid_scores)) if valid_scores else None

    s_norm_str = f"{score_normal:.2f}" if score_normal is not None else "FAIL"
    s_inv_str  = f"{score_inverted:.2f}" if score_inverted is not None else "FAIL"
    s_mit_str  = f"{score_mitigado:.2f}" if score_mitigado is not None else "FAIL"
    print(f"      Score normal: {s_norm_str} | Score invertido: {s_inv_str} | Mitigado: {s_mit_str}")

    position_results.append({
        "doc_id"         : doc["doc_id"],
        "n_gold"         : doc["n_gold"],
        "n_pred"         : len(doc["pred"]),
        "gold"           : doc["gold"],
        "pred"           : doc["pred"],
        "score_normal"   : score_normal,
        "score_inverted" : score_inverted,
        "delta_posicion" : delta,
        "score_mitigado" : score_mitigado,
        "result_normal"  : result_normal,
        "result_inverted": result_inverted,
    })

print("\nListo.")


Corriendo juez (normal + invertido) sobre 30 rich examples...
Total llamadas al juez: 60

  [1/30] es-S0376-78922009000100011-1 | n_gold=28 | n_pred=26
      Score normal: FAIL | Score invertido: FAIL | Mitigado: FAIL
  [2/30] es-S1130-63432015000400006-1 | n_gold=26 | n_pred=27
      Score normal: FAIL | Score invertido: FAIL | Mitigado: FAIL
  [3/30] es-S1130-63432014000100012-1 | n_gold=24 | n_pred=24
      Score normal: FAIL | Score invertido: FAIL | Mitigado: FAIL
  [4/30] S0004-06142007000100012-1 | n_gold=23 | n_pred=25
      Score normal: FAIL | Score invertido: FAIL | Mitigado: FAIL
  [5/30] es-S0210-48062005000300016-2 | n_gold=22 | n_pred=27
      Score normal: FAIL | Score invertido: FAIL | Mitigado: FAIL
  [6/30] es-S1130-01082007000700011-2 | n_gold=22 | n_pred=25
      Score normal: FAIL | Score invertido: FAIL | Mitigado: FAIL
  [7/30] es-S0212-16112012000300028-1 | n_gold=21 | n_pred=21
      Score normal: FAIL | Score invertido: FAIL | Mitigado: FAIL
  [8/30] es-S0004

In [19]:
df_pos = pd.DataFrame(position_results)
deltas = df_pos["delta_posicion"].dropna()

print("=" * 55)
print("ANÁLISIS — SESGO DE POSICIÓN")
print("=" * 55)
print(f"  Ejemplos evaluados          : {len(df_pos)}")
print(f"  Delta medio  |normal-inv|   : {deltas.mean():.3f}")
print(f"  Delta máximo                : {deltas.max():.3f}")
print(f"  Delta mediana               : {deltas.median():.3f}")
print(f"  % ejemplos con delta > 1    : {(deltas > 1).mean()*100:.1f}%")
print()

if deltas.mean() < 0.5:
    verdict = "ESTABLE respecto a posición (delta medio < 0.5). Mitigación es una precaución razonable."
elif deltas.mean() < 1.0:
    verdict = "SESGO MODERADO de posición (delta 0.5–1.0). Mitigación reduce el efecto."
else:
    verdict = "SESGO FUERTE de posición (delta > 1.0). Considerar cambiar el prompt."
print(f"Conclusión: {verdict}")

print("\nTop 5 con mayor delta:")
display(df_pos[["doc_id", "score_normal", "score_inverted", "delta_posicion"]]
        .sort_values("delta_posicion", ascending=False).head(5))


ANÁLISIS — SESGO DE POSICIÓN
  Ejemplos evaluados          : 30
  Delta medio  |normal-inv|   : nan
  Delta máximo                : nan
  Delta mediana               : nan
  % ejemplos con delta > 1    : nan%

Conclusión: SESGO FUERTE de posición (delta > 1.0). Considerar cambiar el prompt.

Top 5 con mayor delta:


,doc_id,score_normal,score_inverted,delta_posicion
0,es-S0376-78922009000100011-1,None,None,None
1,es-S1130-63432015000400006-1,None,None,None
2,es-S1130-63432014000100012-1,None,None,None
3,S0004-06142007000100012-1,None,None,None
4,es-S0210-48062005000300016-2,None,None,None


## 5. Mitigación del sesgo de longitud

### Qué es el sesgo de longitud
El juez puede puntuar más alto respuestas más largas, independientemente de si son correctas.

### Protocolo de detección
Pares de validación con asimetría de longitud controlada:
- **Tipo A** — Correcta corta vs. Incorrecta larga: el juez debería puntuar más alto la correcta.
- **Tipo B** — Correcta larga vs. Incorrecta corta: el juez debería puntuar más alto la correcta.


In [20]:
LENGTH_BIAS_PAIRS = [
    # Tipo A — correcta corta vs. incorrecta larga
    {
        "tipo": "A", "descripcion": "Correcta corta vs. Incorrecta larga",
        "gold": ["neumonía", "sepsis"],
        "pred_correcta" : ["neumonía"],
        "pred_incorrecta": ["hiperglucemia", "dislipidemia", "hipotiroidismo",
                             "artritis reumatoide", "fibromialgia",
                             "esclerosis múltiple", "enfermedad de crohn", "psoriasis"],
    },
    {
        "tipo": "A", "descripcion": "Correcta corta vs. Incorrecta larga",
        "gold": ["infarto agudo de miocardio", "insuficiencia cardiaca", "fibrilación auricular"],
        "pred_correcta" : ["infarto agudo de miocardio"],
        "pred_incorrecta": ["diabetes mellitus tipo 2", "hipertensión arterial",
                             "anemia ferropénica", "neuropatía diabética",
                             "retinopatía diabética", "enfermedad renal crónica", "osteoporosis"],
    },
    # Tipo B — correcta larga vs. incorrecta corta
    {
        "tipo": "B", "descripcion": "Correcta larga vs. Incorrecta corta",
        "gold": ["neumonía", "sepsis", "insuficiencia respiratoria aguda"],
        "pred_correcta" : ["neumonía", "sepsis", "insuficiencia respiratoria aguda"],
        "pred_incorrecta": ["fibromialgia"],
    },
    {
        "tipo": "B", "descripcion": "Correcta larga vs. Incorrecta corta",
        "gold": ["carcinoma ductal infiltrante", "metástasis hepática",
                  "anemia", "trombocitopenia", "neutropenia febril"],
        "pred_correcta" : ["carcinoma ductal infiltrante", "metástasis hepática",
                             "anemia", "trombocitopenia"],
        "pred_incorrecta": ["psoriasis"],
    },
    # Tipo A especial — gold vacío
    {
        "tipo": "A", "descripcion": "Correcta vacía vs. Incorrecta larga (FP puros)",
        "gold": [],
        "pred_correcta" : [],
        "pred_incorrecta": ["artritis", "hipertensión", "diabetes",
                              "anemia", "fibromialgia", "neuropatía"],
    },
]
print(f"Pares preparados: {len(LENGTH_BIAS_PAIRS)}")
for p in LENGTH_BIAS_PAIRS:
    print(f"  Tipo {p['tipo']}: n_correcta={len(p['pred_correcta'])}, n_incorrecta={len(p['pred_incorrecta'])}")


Pares preparados: 5
  Tipo A: n_correcta=1, n_incorrecta=8
  Tipo A: n_correcta=1, n_incorrecta=7
  Tipo B: n_correcta=3, n_incorrecta=1
  Tipo B: n_correcta=4, n_incorrecta=1
  Tipo A: n_correcta=0, n_incorrecta=6


In [21]:
print("Corriendo juez sobre pares de longitud...")
length_results = []

for i, pair in enumerate(LENGTH_BIAS_PAIRS):
    print(f"  [{i+1}/{len(LENGTH_BIAS_PAIRS)}] Tipo {pair['tipo']} — {pair['descripcion']}")
    r_cor = call_judge(pair["gold"], pair["pred_correcta"],  order="normal")
    r_inc = call_judge(pair["gold"], pair["pred_incorrecta"], order="normal")
    sc, si = compute_score(r_cor), compute_score(r_inc)
    length_results.append({
        "tipo"              : pair["tipo"],
        "descripcion"       : pair["descripcion"],
        "n_correcta"        : len(pair["pred_correcta"]),
        "n_incorrecta"      : len(pair["pred_incorrecta"]),
        "score_correcta"    : sc,
        "score_incorrecta"  : si,
        "juez_premia_calidad": (sc > si) if (sc is not None and si is not None) else None,
        "justif_correcta"   : r_cor.get("justificacion", ""),
        "justif_incorrecta" : r_inc.get("justificacion", ""),
    })

print("\nListo.")


Corriendo juez sobre pares de longitud...
  [1/5] Tipo A — Correcta corta vs. Incorrecta larga
  [2/5] Tipo A — Correcta corta vs. Incorrecta larga
  [3/5] Tipo B — Correcta larga vs. Incorrecta corta
  [4/5] Tipo B — Correcta larga vs. Incorrecta corta
  [5/5] Tipo A — Correcta vacía vs. Incorrecta larga (FP puros)

Listo.


In [22]:
df_len  = pd.DataFrame(length_results)
n_ok    = df_len["juez_premia_calidad"].sum()
n_total = df_len["juez_premia_calidad"].notna().sum()

print("=" * 55)
print("ANÁLISIS — SESGO DE LONGITUD")
print("=" * 55)
print(f"  Pares evaluados: {n_total}")
print(f"  El juez premió calidad sobre longitud: {n_ok}/{n_total} ({n_ok/n_total*100:.0f}%)")
print()

if n_ok / n_total >= 0.8:
    print("El juez NO muestra sesgo de longitud relevante.")
elif n_ok / n_total >= 0.6:
    print("Sesgo LEVE de longitud — en algunos casos premia extensión.")
else:
    print("Sesgo FUERTE de longitud — agregar instrucción explícita al prompt.")

print()
display(df_len[["tipo", "descripcion", "n_correcta", "n_incorrecta",
               "score_correcta", "score_incorrecta", "juez_premia_calidad"]])


ANÁLISIS — SESGO DE LONGITUD
  Pares evaluados: 5
  El juez premió calidad sobre longitud: 5/5 (100%)

El juez NO muestra sesgo de longitud relevante.



,tipo,descripcion,n_correcta,n_incorrecta,score_correcta,score_incorrecta,juez_premia_calidad
0,A,Correcta corta vs. Incorrecta larga,1,8,4.50,3.0,True
1,A,Correcta corta vs. Incorrecta larga,1,7,4.25,3.0,True
2,B,Correcta larga vs. Incorrecta corta,3,1,5.00,2.5,True
3,B,Correcta larga vs. Incorrecta corta,4,1,4.75,3.0,True
4,A,Correcta vacía vs. Incorrecta larga (FP puros),0,6,5.00,2.0,True


## 6. Sesgo de auto-preferencia (documentación)

### Qué es
Un LLM juez tiende a preferir outputs de modelos de su misma familia arquitectural o de entrenamiento.

### Situación en este proyecto
- **Juez:** Modelo externo vía Groq (familia Meta / Llama, OpenAI o Qwen)
- **Modelo evaluado:** `roberta-base-biomedical-clinical-es` (encoder RoBERTa, PlanTL / BSC)
- **Modelo alternativo del proyecto:** `mT5` (encoder-decoder, Google)

Al usar un juez LLM externo vía Groq (como Llama, GPT-OSS o Qwen), el juez **no pertenece a la familia de ninguno de los dos modelos del proyecto**, lo cual mitiga el riesgo de auto-preferencia por diseño arquitectónico.

### Mitigaciones aplicadas
1. **Output anonimizado** — el prompt NO menciona el nombre del modelo evaluado.
2. **Formato estandarizado** — BIO → lista para RoBERTa, texto → lista para mT5, presentados exactamente igual.
3. **Independencia de familia** — Juez externo vs. Modelo evaluado RoBERTa.

### Limitación residual
Sin un segundo juez de otra familia distinta (ej. Claude o GPT-4o) para contrastar correlación inter-jueces, se reporta como limitación metodológica.


In [23]:
sample_prompt = build_judge_prompt(["neumonía", "sepsis"], ["neumonía"])
forbidden     = ["roberta", "bert", "mt5", "llama", "gpt", "gemini", "plantl", "biomedical", "clinical"]
found = [n for n in forbidden if n.lower() in sample_prompt.lower()]

if not found:
    print("El prompt NO menciona ningún modelo — output completamente anonimizado.")
else:
    print(f"ATENCIÓN: el prompt menciona {found} — revisar RUBRICA_TEMPLATE.")

print()
print("Resumen de mitigaciones de auto-preferencia:")
print("  1. Output anonimizado (sin nombres de modelo en el prompt): OK")
print("  2. Formato estandarizado (lista -> str uniforme): OK")
print(f"  3. Familia cruzada independiente (Juez: {JUDGE_MODEL} vs. RoBERTa): OK")


El prompt NO menciona ningún modelo — output completamente anonimizado.

Resumen de mitigaciones de auto-preferencia:
  1. Output anonimizado (sin nombres de modelo en el prompt): OK
  2. Formato estandarizado (lista -> str uniforme): OK
  3. Familia cruzada independiente (Juez: openai/gpt-oss-120b vs. RoBERTa): OK


## 7. Scorecard final sobre rich examples

Usamos el **score mitigado** (promedio normal + invertido, Sección 4) como score final del juez.


In [24]:
def compute_exact_f1_doc(gold: list, pred: list) -> dict:
    """F1 exact-match para un documento (para triangular con la métrica de Luis)."""
    g, p = set(gold), set(pred)
    tp   = len(g & p)
    fp   = len(p - g)
    fn   = len(g - p)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) else 0.0
    return {"precision": prec, "recall": rec, "f1": f1}


scorecard_rows = []
for row in position_results:
    exact = compute_exact_f1_doc(row["gold"], row["pred"])
    scorecard_rows.append({
        "doc_id"          : row["doc_id"],
        "n_gold_entities" : row["n_gold"],
        "n_pred_entities" : row["n_pred"],
        "score_juez"      : row["score_mitigado"],
        "delta_posicion"  : row["delta_posicion"],
        "f1_exacto"       : exact["f1"],
        "precision_exacta": exact["precision"],
        "recall_exacto"   : exact["recall"],
        "score_semantico" : None,   # placeholder — tarea de Pau
        "gold"            : row["gold"],
        "pred"            : row["pred"],
        "justificacion"   : row["result_normal"].get("justificacion", ""),
    })

df_sc = pd.DataFrame(scorecard_rows)

print("=" * 55)
print("SCORECARD — Rich Examples")
print("=" * 55)
print(f"  Docs evaluados           : {len(df_sc)}")
print(f"  Score juez (mitigado)    : {df_sc['score_juez'].mean():.3f} ± {df_sc['score_juez'].std():.3f}")
print(f"  F1 exacto (mismo subset) : {df_sc['f1_exacto'].mean():.3f} ± {df_sc['f1_exacto'].std():.3f}")
print()
display(df_sc[["doc_id", "n_gold_entities", "n_pred_entities",
               "score_juez", "delta_posicion",
               "f1_exacto", "precision_exacta", "recall_exacto"]])


SCORECARD — Rich Examples
  Docs evaluados           : 30
  Score juez (mitigado)    : nan ± nan
  F1 exacto (mismo subset) : 0.736 ± 0.102



,doc_id,n_gold_entities,n_pred_entities,score_juez,delta_posicion,f1_exacto,precision_exacta,recall_exacto
0,es-S0376-78922009000100011-1,28,26,None,None,0.703704,0.730769,0.678571
1,es-S1130-63432015000400006-1,26,27,None,None,0.716981,0.703704,0.730769
2,es-S1130-63432014000100012-1,24,24,None,None,0.750000,0.750000,0.750000
3,S0004-06142007000100012-1,23,25,None,None,0.708333,0.680000,0.739130
4,es-S0210-48062005000300016-2,22,27,None,None,0.816327,0.740741,0.909091
5,es-S1130-01082007000700011-2,22,25,None,None,0.765957,0.720000,0.818182
6,es-S0212-16112012000300028-1,21,21,None,None,0.666667,0.666667,0.666667
7,es-S0004-06142009000900012-1,20,22,None,None,0.904762,0.863636,0.950000
8,es-S1137-66272007000300014-1,20,21,None,None,0.878049,0.857143,0.900000
9,es-S0212-71992005001200010-1,20,23,None,None,0.697674,0.652174,0.750000


## 8. Interpretación — ¿Qué revela el juez que el F1 exacto no capta?

| Patrón | F1 exacto | Score juez | Interpretación |
|--------|-----------|-----------|----------------|
| **A** | ≈ 0 | ≥ 3.5 | Entendió la entidad pero con **boundary distinto** — error de span, no de comprensión. |
| **B** | > 0.3 | ≤ 2.5 | Acertó la string exacta pero el output **no es clínicamente satisfactorio**. |
| **C** | ≈ 0 | ≤ 2.0 | Fallo completo. |
| **D** | ≥ 0.7 | ≥ 4.0 | Funcionamiento correcto — referencia positiva. |
| **E** | resto | resto | Caso mixto. |


In [25]:
def classify_pattern(row):
    s, f = row["score_juez"], row["f1_exacto"]
    if s is None or f is None: return "? — datos faltantes"
    if f < 0.1 and s >= 3.5:   return "A — boundary error (comprensión OK, span malo)"
    if f > 0.3 and s <= 2.5:   return "B — string OK, calidad clínica baja"
    if f < 0.1 and s <= 2.0:   return "C — fallo completo"
    if f >= 0.7 and s >= 4.0:  return "D — referencia positiva"
    return "E — caso mixto"


df_sc["patron"] = df_sc.apply(classify_pattern, axis=1)
print("Distribución de patrones:")
print(df_sc["patron"].value_counts().to_string())

for patron_key, label in [("A", "Boundary error, comprensión OK"),
                           ("B", "String OK, calidad clínica baja")]:
    sub = df_sc[df_sc["patron"].str.startswith(patron_key)]
    print(f"\n--- Patrón {patron_key} ({len(sub)} docs) — {label} ---")
    for _, r in sub.iterrows():
        print(f"  doc: {r['doc_id'][:45]:45s} | F1={r['f1_exacto']:.2f} | Juez={r['score_juez']:.2f}")
        print(f"    Gold: {r['gold']}")
        print(f"    Pred: {r['pred']}")


Distribución de patrones:
patron
? — datos faltantes    30

--- Patrón A (0 docs) — Boundary error, comprensión OK ---

--- Patrón B (0 docs) — String OK, calidad clínica baja ---


In [26]:
import json as _j

OUTPUT_DIR = BASE_DIR / "outputs" / "M2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "llm_judge_scorecard.csv"
df_sc.drop(columns=["gold", "pred"], errors="ignore").to_csv(
    csv_path, index=False, encoding="utf-8"
)
print(f"Scorecard  -> {csv_path}")

bias_summary = {
    "judge_model"        : JUDGE_MODEL,
    "judge_provider"     : "Groq",
    "base_model"         : BASE_CHECKPOINT,
    "seed"               : SEED,
    "n_rich_examples"    : len(df_sc),
    "min_unique_entities": MIN_UNIQUE_ENTITIES,
    "score_juez_mean"    : float(df_sc["score_juez"].mean()),
    "score_juez_std"     : float(df_sc["score_juez"].std()),
    "f1_exacto_mean"     : float(df_sc["f1_exacto"].mean()),
    "position_bias": {
        "delta_mean" : float(deltas.mean()),
        "delta_max"  : float(deltas.max()),
        "mitigation" : "average of normal and inverted order scores",
    },
    "length_bias": {
        "pairs_quality_wins": int(n_ok),
        "pairs_total"       : int(n_total),
        "pct_quality_wins"  : float(n_ok / n_total),
        "mitigation"        : "manual validation pairs — judge follows content, not length",
    },
    "auto_preference_bias": {
        "output_anonymized"   : True,
        "format_standardized" : True,
        "cross_family_test"   : False,
        "note": "Cannot quantify without a judge from a different model family",
    },
    "pattern_distribution": df_sc["patron"].value_counts().to_dict(),
}

json_path = OUTPUT_DIR / "llm_judge_bias_summary.json"
with open(json_path, "w", encoding="utf-8") as f:
    _j.dump(bias_summary, f, indent=2, ensure_ascii=False)
print(f"Bias summary -> {json_path}")

print("\n" + "=" * 55)
print("RESUMEN FINAL")
print("=" * 55)
print(f"  Sesgo posición  — delta medio: {deltas.mean():.3f}  | mitigación: promedio normal+invertido")
print(f"  Sesgo longitud  — calidad gana: {n_ok}/{n_total}    | mitigación: pares de validación")
print(f"  Auto-preferencia— anonimizado: OK | test cross-family: no disponible (limitación)")
print(f"  Score juez final (mitigado)  : {df_sc['score_juez'].mean():.3f}")
print(f"  F1 exacto (mismo subset)     : {df_sc['f1_exacto'].mean():.3f}")


Scorecard  -> /content/drive/MyDrive/TopicosIA/outputs/M2/llm_judge_scorecard.csv
Bias summary -> /content/drive/MyDrive/TopicosIA/outputs/M2/llm_judge_bias_summary.json

RESUMEN FINAL
  Sesgo posición  — delta medio: nan  | mitigación: promedio normal+invertido
  Sesgo longitud  — calidad gana: 5/5    | mitigación: pares de validación
  Auto-preferencia— anonimizado: OK | test cross-family: no disponible (limitación)
  Score juez final (mitigado)  : nan
  F1 exacto (mismo subset)     : 0.736
